### Set up drive

In [52]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [53]:
%cd /content/drive/MyDrive/Learning/UA_MSIS/Courses/10_CapstoneProject/MURA/rag

/content/drive/MyDrive/Learning/UA_MSIS/Courses/10_CapstoneProject/MURA/rag


In [54]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Learning/UA_MSIS/Courses/10_CapstoneProject/MURA"
)
RAG_DIR = PROJECT_ROOT / "rag"
DATA_DIR = PROJECT_ROOT / "data"

sys.path.append(str(PROJECT_ROOT))

PROJECT_ROOT, RAG_DIR, DATA_DIR

(PosixPath('/content/drive/MyDrive/Learning/UA_MSIS/Courses/10_CapstoneProject/MURA'),
 PosixPath('/content/drive/MyDrive/Learning/UA_MSIS/Courses/10_CapstoneProject/MURA/rag'),
 PosixPath('/content/drive/MyDrive/Learning/UA_MSIS/Courses/10_CapstoneProject/MURA/data'))

### Import

In [ ]:
!pip install -q transformers sentence-transformers pandas pyarrow chromadb

In [55]:
from transformers import logging

logging.set_verbosity_error()

In [56]:
import pandas as pd
from tqdm import tqdm

from rag.config import RAGConfig
from rag.external_corpus import ExternalCorpus
from rag.retriever import VectorRAGRetriever
from rag.vector_store import ChromaVectorStore
from rag.local_corpus import LocalCorpus

In [57]:
import importlib
import rag.vector_store
import rag.retriever
import rag.external_corpus

importlib.reload(rag.vector_store)
importlib.reload(rag.retriever)
importlib.reload(rag.external_corpus)

<module 'rag.external_corpus' from '/content/drive/MyDrive/Learning/UA_MSIS/Courses/10_CapstoneProject/MURA/rag/external_corpus.py'>

### Load data

In [58]:
import torch

print("GPU available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

GPU available: True
GPU: NVIDIA A100-SXM4-40GB


In [59]:
csv_path = DATA_DIR / "finance/data.csv"
data = pd.read_csv(csv_path)

# Ensure datetime
data["published_at"] = pd.to_datetime(data["published_at"])

len(data)

2291

### Indexing

In [60]:
queries = [
    f"Ticker: {row['ticker']}. Headline: {row['title']}" for _, row in data.iterrows()
]

In [61]:
from sentence_transformers import SentenceTransformer

config = RAGConfig()

query_embedder = SentenceTransformer(config.embedding_model, device="cuda")

query_embeddings = query_embedder.encode(
    queries, batch_size=64, show_progress_bar=True, convert_to_numpy=True
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/36 [00:00<?, ?it/s]

In [62]:
external_corpus = ExternalCorpus(PROJECT_ROOT / "rag/financial_news_2023.parquet")

In [63]:
vector_store = ChromaVectorStore(config.embedding_model)

texts, metadatas = external_corpus.get_all_texts_and_metadata()

vector_store.build_global_index(texts, metadatas)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Adding to Chroma: 100%|██████████| 1984/1984 [1:12:19<00:00,  2.19s/it]


### Retrieval

In [64]:
retriever = VectorRAGRetriever(config=config, vector_store=vector_store)

In [65]:
# Instantiate once
local_corpus = LocalCorpus()

retrieved_contexts = []

for emb, (_, row) in tqdm(zip(query_embeddings, data.iterrows()), total=len(data)):
    published_at = pd.to_datetime(row["published_at"])
    date = published_at.date().isoformat()

    # External corpus (vector search)
    context = retriever.retrieve(query_embedding=emb, date=date)

    retrieved_contexts.append(context)

100%|██████████| 2291/2291 [14:39<00:00,  2.61it/s]


### Save context to dataset

In [83]:
data["external_context_1"] = [ctx[1].strip() for ctx in retrieved_contexts]
data["external_context_2"] = [ctx[2].strip() for ctx in retrieved_contexts]
data["external_context_3"] = [ctx[3].strip() for ctx in retrieved_contexts]

In [84]:
out_path = DATA_DIR / "finance/data_with_context.csv"
data.to_csv(out_path, index=False)

print("Saved to:", out_path)

Saved to: /content/drive/MyDrive/Learning/UA_MSIS/Courses/10_CapstoneProject/MURA/data/finance/data_with_context.csv


In [85]:
data.head()

,published_at,ticker,true_sentiment,title,author,url,source,text,finbert_sentiment,finbert_sent_score,external_context_1,external_context_2,external_context_3
0,2023-01-12 07:47:00,EURCHF,Positive,Euro to benefit from the ECBs pronounced hawki...,FXStreet Insights Team,https://www.fxstreet.com/news/euro-to-benefit-...,FX Street,The Euro was able to appreciate particularly s...,Positive,0.85,CEE MARKETS-FX drifts before U.S. inflation da...,European shares rise as investors await U.S. i...,MOVES-Rothschild & Co appoint Horn as capital ...
1,2023-01-12 10:34:00,EURCHF,Positive,EURCHF Trend higher may remain in place – ING,FXStreet Insights Team,https://www.fxstreet.com/news/eur-chf-trend-hi...,FX Street,EUR/CHF yesterday broke above 1.00. Economists...,Positive,0.51,European shares rise as investors await U.S. i...,CEE MARKETS-FX drifts before U.S. inflation da...,European shares rise as investors await U.S. i...
2,2023-01-12 11:40:00,EURCHF,Neutral,Does a jump in EURCHF point to a break above 1...,FXStreet Insights Team,https://www.fxstreet.com/news/does-a-jump-in-e...,FX Street,EUR/CHF vaults parity for the first time since...,Neutral,0.37,"Down -29.05% in 4 Weeks, Here's Why You Should...",CEE MARKETS-FX drifts before U.S. inflation da...,Bear Market Watch: What Will Mean a Bull Marke...
3,2023-01-12 15:32:00,EURCHF,Positive,EURCHF could extend its advance back to levels...,FXStreet Insights Team,https://www.fxstreet.com/news/eur-chf-could-ex...,FX Street,EUR/CHF climbs back above parity. Economists a...,Positive,0.64,CEE MARKETS-FX drifts before U.S. inflation da...,Bullish Two Hundred Day Moving Average Cross -...,"SPSM, AMN, FN, SPSC: ETF Inflow Alert\n\nLooki..."
4,2023-01-13 11:37:00,EURCHF,Positive,EURCHF to head higher towards 10130 and projec...,FXStreet Insights Team,https://www.fxstreet.com/news/eur-chf-to-head-...,FX Street,EUR/CHF has broken out above the sideways rang...,Positive,0.83,CEE MARKETS-Forint climbs back near 5-month hi...,"SOXS, EGPT: Big ETF Inflows\n\nComparing units...","SOXS, EGPT: Big ETF Inflows\n\nComparing units..."
